In [17]:
import os
import sys
import re
import numpy as np
import pandas as pd
import plotly.express as px
import scipy.stats as stats
from scipy.stats import pearsonr
from dash import html, dcc, Input, Output, Dash
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append(os.path.join(os.getcwd(), '..', '..', '..'))
from baseVR.base_functionality import init_import_paths
init_import_paths() 

from CustomLogger import CustomLogger as Logger
from analytics_processing import analytics

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args, fullfnames2snames
from dashsrc.plot_components.plots import plot_TrackFiringRate
from dashsrc.plot_components.plots import plot_unit_fr_stability


In [18]:
Logger().init_logger(None, None, logging_level="DEBUG")
animal_ids = [6]
paradigm = [1100]
session_range = [1,33]
session_ids = None
normalize = True
smooth = False
excl_session_names =  ['2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min', '2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min', '2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min', '2024-12-11_17-42_rYL006_P1100_LinearTrackStop_30min'] # ignore short sessions (10, 24,25)

session_dirs = sessionlist_fullfnames_from_args(paradigm, animal_ids, session_ids, excl_session_names=excl_session_names)[0]
session_names= fullfnames2snames(session_dirs)

2026-03-02 16:22:02,929|DEBUG|53778|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Searching NAS for applicable sessions...
Searching NAS for applicable sessions...================================================================================

2026-03-02 16:22:02,929|DEBUG|53778|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Searching NAS for applicable sessions...
2026-03-02 16:22:02,965|DEBUG|53778|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min.hdf5 excluded
Session 2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min.hdf5 excluded================================================================================

2026-03-02 16:22:02,965|DEBUG|53778|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min.hdf5 excluded
2026-03-02 16:22:02,968|DEBUG|53778|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-12-11_17-42_rYL006_P1100_Linea

In [19]:
# firing rates and behavior data
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
# fr_track = analytics.get_analytics('FiringRateTrackwiseHz', session_names=session_names)
t0_events = analytics.get_analytics('TrialWiseT0Events40ms', session_names=session_names)
fr_z_scored  = analytics.get_analytics('FiringRate40msZ', session_names=session_names)
fr_z_all_sess = fr.apply(lambda unit_fr: ((unit_fr - unit_fr.mean()) / unit_fr.std()))
# behav = analytics.get_analytics('BehaviorTrackwise', session_names=session_names)
# behav.index = behav.index.droplevel(('animal_id', 'paradigm_id', 'entry_id'))

2026-03-02 16:22:03,002|DEBUG|53778|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
Inferring NAS paths for list of session names...================================================================================

2026-03-02 16:22:03,002|DEBUG|53778|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-03-02 16:22:03,358|DEBUG|53778|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-03-02 16:22:03,358|DEBUG|53778|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-03-02 16:22:03,360|DEBUG|53778|analytics|get_analytics
	Processing Firi

In [20]:
t0_events

trial_id  cue  trial_outcome  \
paradigm_id animal_id session_id       entry_id                                 
1100        6         2024-11-14_16-40 0              1.0  1.0            0.0   
                                       1              1.0  1.0            0.0   
                                       2              1.0  1.0            0.0   
                                       3              1.0  1.0            0.0   
                                       4              1.0  1.0            0.0   
...                                                   ...  ...            ...   
                      2025-01-27_13-39 1104         152.0  2.0            1.0   
                                       1105         152.0  2.0            1.0   
                                       1106         152.0  2.0            1.0   
                                       1107         152.0  2.0            1.0   
                                       1108         152.0  2.0            1.0   

                                                 choice_R1  choice_R2  \
paradigm_id animal_id session_id       entry_id                         
1100        6         2024-11-14_16-40 0               0.0        0.0   
                                       1               0.0        0.0   
                                       2               0.0        0.0   
                                       3               0.0        0.0   
                                       4               0.0        0.0   
...                                                    ...        ...   
                      2025-01-27_13-39 1104            0.0        1.0   
                                       1105            0.0        1.0   
                                       1106            0.0        1.0   
                                       1107            0.0        1.0   
                                       1108            0.0        1.0   

                                                     t0_event_name  \
paradigm_id animal_id session_id       entry_id                      
1100        6         2024-11-14_16-40 0           cueZone_visible   
                                       1             cueZone_entry   
                                       2              cueZone_exit   
                                       3         enter_reward1Zone   
                                       4         enter_reward2Zone   
...                                                            ...   
                      2025-01-27_13-39 1104           cueZone_exit   
                                       1105      enter_reward1Zone   
                                       1106      enter_reward2Zone   
                                       1107       exit_reward1Zone   
                                       1108       exit_reward2Zone   

                                                         t0  x_position  \
paradigm_id animal_id session_id       entry_id                           
1100        6         2024-11-14_16-40 0            4600000 -119.732067   
                                       1            5400000  -79.536247   
                                       2            6760000   24.898020   
                                       3            7080000   51.438920   
                                       4            9480000  170.376450   
...                                                     ...         ...   
                      2025-01-27_13-39 1104      4396000000   25.843260   
                                       1105      4396480000   49.770475   
                                       1106      4398880000  170.561500   
                                       1107      4397720000  109.711900   
                                       1108      4401200000  229.449700   

                                                 x_alignment  \
paradigm_id animal_id session_id       entry_id                
1100        6         2024-11-

In [21]:
# Event-based neuron classification: cue / choice / reward_sound (+ detailed event regions)
# Tests whether units fire more during event intervals than during the rest of session time.

N_SHUFFLES = 400
THRESHOLD_PERCENTILE = 99
ALPHA = 0.01
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

if fr is None or t0_events is None:
    raise ValueError("`fr` and `t0_events` must be loaded before running this cell.")


def _parse_interval(value):
    if isinstance(value, pd.Interval):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    if isinstance(value, dict):
        left = value.get("left", np.nan)
        right = value.get("right", np.nan)
        if pd.notna(left) and pd.notna(right):
            return pd.Interval(float(left), float(right), closed="right")
        return np.nan
    if isinstance(value, (tuple, list)) and len(value) >= 2:
        left, right = value[0], value[1]
        if pd.notna(left) and pd.notna(right):
            return pd.Interval(float(left), float(right), closed="right")
        return np.nan

    txt = str(value).strip()
    if txt.lower() in {"", "nan", "none", "nat"}:
        return np.nan

    m = re.match(r"^[\(\[]\s*([-+0-9eE\.]+)\s*,\s*([-+0-9eE\.]+)\s*[\)\]]$", txt)
    if m:
        return pd.Interval(float(m.group(1)), float(m.group(2)), closed="right")

    return np.nan


def _safe_divide(num, den):
    return np.divide(
        num,
        den,
        out=np.full_like(num, np.nan, dtype=float),
        where=np.asarray(den) > 0,
    )


def _compute_diff_from_idx(x_data, idx, total_sum, total_cnt, valid_mask):
    n_rows = x_data.shape[0]
    if idx.size == 0 or idx.size >= n_rows:
        return np.full(x_data.shape[1], np.nan, dtype=float)

    in_sum = x_data[idx, :].sum(axis=0)
    if valid_mask is None:
        in_cnt = np.full(x_data.shape[1], idx.size, dtype=float)
    else:
        in_cnt = valid_mask[idx, :].sum(axis=0)

    out_sum = total_sum - in_sum
    out_cnt = total_cnt - in_cnt

    in_mean = _safe_divide(in_sum, in_cnt)
    out_mean = _safe_divide(out_sum, out_cnt)
    return in_mean - out_mean


fr_flat = fr.reset_index().copy()
t0_flat = t0_events.reset_index().copy()

if "session_id" not in fr_flat.columns or "session_id" not in t0_flat.columns:
    raise KeyError("Both `fr` and `t0_events` must contain `session_id`.")

if {"from_ephys_timestamp", "to_ephys_timestamp"}.issubset(fr_flat.columns):
    fr_flat["__time_mid"] = (
        pd.to_numeric(fr_flat["from_ephys_timestamp"], errors="coerce")
        + pd.to_numeric(fr_flat["to_ephys_timestamp"], errors="coerce")
    ) / 2.0
elif "from_ephys_timestamp" in fr_flat.columns:
    fr_flat["__time_mid"] = pd.to_numeric(fr_flat["from_ephys_timestamp"], errors="coerce")
else:
    raise KeyError("`fr` needs `from_ephys_timestamp` (and optionally `to_ephys_timestamp`).")

fr_flat = fr_flat.dropna(subset=["session_id", "__time_mid"]).copy()
unit_cols = [c for c in fr_flat.columns if str(c).startswith("Unit")]
if len(unit_cols) == 0:
    raise ValueError("No unit columns found in `fr` (expected names like `Unit0001`).")

trial_col = next((c for c in ["trial_id", "triald_id"] if c in t0_flat.columns), None)
if trial_col is None:
    trial_col = "trial_id"
    t0_flat[trial_col] = np.nan

raw_interval_candidates = {
    "cue_entry": ["cue_entry_interval"],
    "reward1_sound": ["reward1_sound_interval", "reward1_valve_open_interval"],
    "reward2_sound": ["reward2_sound_interval", "reward2_valve_open_interval"],
    "R1_entry": ["R1_entry_interval"],
    "R2_entry": ["R2_entry_interval"],
}

resolved_interval_cols = {}
for key, candidates in raw_interval_candidates.items():
    col = next((c for c in candidates if c in t0_flat.columns), None)
    if col is not None:
        resolved_interval_cols[key] = col

required_core = {"cue_entry", "R1_entry", "R2_entry"}
missing_core = sorted(required_core.difference(resolved_interval_cols))
if missing_core:
    raise KeyError(f"Missing required interval columns in t0_events: {missing_core}")

if "reward1_sound" not in resolved_interval_cols and "reward2_sound" not in resolved_interval_cols:
    print("Warning: no reward sound interval columns found (reward sound features will be empty).")

parsed_col_map = {}
for key, col in resolved_interval_cols.items():
    parsed_col = f"__parsed_{key}"
    t0_flat[parsed_col] = t0_flat[col].map(_parse_interval)
    parsed_col_map[key] = parsed_col

session_ids = pd.Index(fr_flat["session_id"].dropna().unique()).intersection(
    pd.Index(t0_flat["session_id"].dropna().unique())
)
if len(session_ids) == 0:
    raise ValueError("No overlapping `session_id` values between `fr` and `t0_events`.")

records = []
timing_records = []
timing_mismatch_examples = []

for session_id in session_ids:
    fr_s = (
        fr_flat[fr_flat["session_id"] == session_id]
        .sort_values("__time_mid")
        .reset_index(drop=True)
    )
    t0_s = t0_flat[t0_flat["session_id"] == session_id].copy()

    if fr_s.empty or t0_s.empty:
        continue

    time_mid = fr_s["__time_mid"].to_numpy(dtype=float)
    n_rows = len(fr_s)
    n_units = len(unit_cols)

    raw_masks = {}
    for key, parsed_col in parsed_col_map.items():
        mask = np.zeros(n_rows, dtype=bool)
        valid_rows = t0_s.loc[t0_s[parsed_col].notna(), [trial_col, parsed_col]].copy()

        n_intervals = len(valid_rows)
        n_matched = 0

        for i, intvl in enumerate(valid_rows[parsed_col].to_numpy()):
            left = float(intvl.left)
            right = float(intvl.right)
            lo = int(np.searchsorted(time_mid, left, side="left"))
            hi = int(np.searchsorted(time_mid, right, side="right"))

            if hi > lo:
                mask[lo:hi] = True
                n_matched += 1
            else:
                if len(timing_mismatch_examples) < 30:
                    timing_mismatch_examples.append(
                        {
                            "session_id": session_id,
                            "interval_key": key,
                            "trial_id": valid_rows.iloc[i][trial_col],
                            "interval_left": left,
                            "interval_right": right,
                            "fr_min_time": float(time_mid.min()),
                            "fr_max_time": float(time_mid.max()),
                        }
                    )

        raw_masks[key] = mask
        timing_records.append(
            {
                "session_id": session_id,
                "interval_key": key,
                "interval_column": resolved_interval_cols[key],
                "n_intervals_total": int(n_intervals),
                "n_intervals_matched": int(n_matched),
                "n_intervals_unmatched": int(n_intervals - n_matched),
                "pct_intervals_matched": float(100 * n_matched / max(n_intervals, 1)),
                "n_bins_covered": int(mask.sum()),
                "n_bins_session": int(n_rows),
                "fr_time_min": float(time_mid.min()),
                "fr_time_max": float(time_mid.max()),
            }
        )

    feature_masks = {
        "cue": raw_masks.get("cue_entry", np.zeros(n_rows, dtype=bool)),
        "choice": (
            raw_masks.get("R1_entry", np.zeros(n_rows, dtype=bool))
            | raw_masks.get("R2_entry", np.zeros(n_rows, dtype=bool))
        ),
        "reward_sound": (
            raw_masks.get("reward1_sound", np.zeros(n_rows, dtype=bool))
            | raw_masks.get("reward2_sound", np.zeros(n_rows, dtype=bool))
        ),
        "choice_r1": raw_masks.get("R1_entry", np.zeros(n_rows, dtype=bool)),
        "choice_r2": raw_masks.get("R2_entry", np.zeros(n_rows, dtype=bool)),
        "reward_sound_r1": raw_masks.get("reward1_sound", np.zeros(n_rows, dtype=bool)),
        "reward_sound_r2": raw_masks.get("reward2_sound", np.zeros(n_rows, dtype=bool)),
    }
    feature_order = list(feature_masks.keys())
    feature_idx = {k: np.flatnonzero(v) for k, v in feature_masks.items()}

    x_raw = fr_s[unit_cols].to_numpy(dtype=float, copy=True)
    has_nan = np.isnan(x_raw).any()
    if has_nan:
        valid_mask = np.isfinite(x_raw)
        x_data = np.where(valid_mask, x_raw, 0.0)
        total_sum = x_data.sum(axis=0)
        total_cnt = valid_mask.sum(axis=0).astype(float)
    else:
        valid_mask = None
        x_data = x_raw
        total_sum = x_data.sum(axis=0)
        total_cnt = np.full(n_units, n_rows, dtype=float)

    observed = {}
    shuffled = {k: np.full((N_SHUFFLES, n_units), np.nan, dtype=float) for k in feature_order}

    for key in feature_order:
        observed[key] = _compute_diff_from_idx(
            x_data=x_data,
            idx=feature_idx[key],
            total_sum=total_sum,
            total_cnt=total_cnt,
            valid_mask=valid_mask,
        )

    if n_rows > 1:
        for s in range(N_SHUFFLES):
            shift = int(rng.integers(0, n_rows))
            for key in feature_order:
                base_idx = feature_idx[key]
                if base_idx.size == 0 or base_idx.size >= n_rows:
                    continue
                shifted_idx = (base_idx + shift) % n_rows
                shuffled[key][s, :] = _compute_diff_from_idx(
                    x_data=x_data,
                    idx=shifted_idx,
                    total_sum=total_sum,
                    total_cnt=total_cnt,
                    valid_mask=valid_mask,
                )

    for u_i, unit in enumerate(unit_cols):
        rec = {
            "session_id": session_id,
            "unit": unit,
            "n_rows_session": int(n_rows),
        }

        for key in feature_order:
            rec[f"n_rows_{key}"] = int(feature_idx[key].size)

            obs = observed[key][u_i]
            sh = shuffled[key][:, u_i]
            valid = np.isfinite(sh)

            if valid.any() and np.isfinite(obs):
                thr = float(np.percentile(sh[valid], THRESHOLD_PERCENTILE))
                pval = float((np.sum(sh[valid] >= obs) + 1) / (valid.sum() + 1))
            else:
                thr = np.nan
                pval = np.nan

            rec[f"obs_{key}"] = float(obs) if np.isfinite(obs) else np.nan
            rec[f"thr99_{key}"] = thr
            rec[f"p_{key}"] = pval
            rec[f"is_{key}_cell"] = bool(np.isfinite(obs) and np.isfinite(thr) and (obs > thr))

        labels = []
        if rec["is_cue_cell"]:
            labels.append("cue")
        if rec["is_choice_cell"]:
            labels.append("choice")
        if rec["is_reward_sound_cell"]:
            labels.append("reward_sound")
        rec["assigned_label"] = "+".join(labels) if labels else "none"

        records.append(rec)

if len(timing_records) == 0:
    raise RuntimeError("No timing records were created. Check session_id overlap and interval parsing.")
if len(records) == 0:
    raise RuntimeError("No classification rows were created. Check interval parsing and FR/event timing alignment.")

classification_eventwise = (
    pd.DataFrame(records)
    .sort_values(["session_id", "unit"])
    .reset_index(drop=True)
)

timing_sanity = pd.DataFrame(timing_records)
timing_summary = (
    timing_sanity.groupby("interval_key", as_index=False)
    .agg(
        n_sessions=("session_id", "nunique"),
        n_intervals_total=("n_intervals_total", "sum"),
        n_intervals_matched=("n_intervals_matched", "sum"),
        n_intervals_unmatched=("n_intervals_unmatched", "sum"),
        n_bins_covered=("n_bins_covered", "sum"),
    )
)
timing_summary["pct_intervals_matched"] = (
    100 * timing_summary["n_intervals_matched"] / timing_summary["n_intervals_total"].clip(lower=1)
)

timing_mismatches = pd.DataFrame(timing_mismatch_examples)

print("Resolved interval columns:", resolved_interval_cols)
print(
    f"Timing sanity check across {timing_sanity['session_id'].nunique()} sessions "
    f"(FR rows: {len(fr_flat):,}, t0 rows: {len(t0_flat):,})"
)
display(timing_summary.sort_values("interval_key"))
if not timing_mismatches.empty:
    print("Examples of unmatched event intervals (first 30):")
    display(timing_mismatches)

broad_flags = ["is_cue_cell", "is_choice_cell", "is_reward_sound_cell"]
detail_flags = [
    "is_choice_r1_cell",
    "is_choice_r2_cell",
    "is_reward_sound_r1_cell",
    "is_reward_sound_r2_cell",
]

print(
    f"Computed classification for {len(classification_eventwise)} unit-session entries "
    f"({classification_eventwise['unit'].nunique()} units, "
    f"{classification_eventwise['session_id'].nunique()} sessions)."
)
display(classification_eventwise[broad_flags + detail_flags].sum().rename("n_unit_sessions").to_frame())
display(classification_eventwise.head())

# Trackwise-like plots (renamed reward_approach -> choice)
dfc = classification_eventwise.copy()

features = ["cue", "choice", "reward_sound"]
obs_cols = [f"obs_{f}" for f in features]
p_cols = [f"p_{f}" for f in features]

obs = dfc[obs_cols].to_numpy(dtype=float)
pvals = dfc[p_cols].to_numpy(dtype=float)

obs_filled = np.where(np.isfinite(obs), obs, -np.inf)
has_obs = np.isfinite(obs).any(axis=1)

pref_idx = np.argmax(obs_filled, axis=1)
pref_obs = np.where(has_obs, obs[np.arange(len(dfc)), pref_idx], np.nan)

runner = obs_filled.copy()
runner[np.arange(len(dfc)), pref_idx] = -np.inf
runner_obs = np.max(runner, axis=1)
runner_obs = np.where(has_obs & np.isfinite(runner_obs), runner_obs, np.nan)

selectivity_index = (pref_obs - runner_obs) / (
    np.abs(pref_obs) + np.abs(runner_obs) + 1e-12
)

pref_p = np.where(has_obs, pvals[np.arange(len(dfc)), pref_idx], np.nan)
shuffle_confidence = 1 - pref_p
is_significant_pref = pref_p < ALPHA
preferred_feature = np.where(
    has_obs,
    np.array(features, dtype=object)[pref_idx],
    "none",
)

certainty_df = dfc[["session_id", "unit"]].copy()
certainty_df["preferred_feature"] = preferred_feature
certainty_df["pref_obs"] = pref_obs
certainty_df["runner_obs"] = runner_obs
certainty_df["selectivity_index"] = selectivity_index
certainty_df["pref_p"] = pref_p
certainty_df["shuffle_confidence"] = shuffle_confidence
certainty_df["certainty_score"] = certainty_df["selectivity_index"] * certainty_df["shuffle_confidence"]
certainty_df["is_significant_pref"] = is_significant_pref

print(
    f"unit-session rows: {len(certainty_df)} | unique units: {certainty_df['unit'].nunique()} | "
    f"unique sessions: {certainty_df['session_id'].nunique()}"
)

fig_certainty = px.scatter(
    certainty_df,
    x="selectivity_index",
    y="shuffle_confidence",
    color="preferred_feature",
    symbol="is_significant_pref",
    opacity=0.7,
    hover_data={
        "session_id": True,
        "unit": True,
        "pref_obs": ":.4f",
        "runner_obs": ":.4f",
        "pref_p": ":.4g",
        "certainty_score": ":.4f",
    },
    title="Event-encoding certainty per unit-session (selectivity vs shuffle confidence)",
    labels={
        "selectivity_index": "Selectivity vs next-best event",
        "shuffle_confidence": "Shuffle confidence (1 - p)",
        "preferred_feature": "Preferred encoding",
        "is_significant_pref": "p < 0.01",
    },
)
fig_certainty.update_layout(template="plotly_white")
fig_certainty.show()

flag_map = {
    "is_cue_cell": "cue",
    "is_choice_cell": "choice",
    "is_reward_sound_cell": "reward_sound",
}
session_frac = (
    dfc.groupby("session_id", as_index=False)[list(flag_map.keys())]
    .mean()
    .rename(columns=flag_map)
)

session_long = session_frac.melt(
    id_vars="session_id",
    var_name="cell_type",
    value_name="fraction_units",
)
session_long["percent_units"] = 100.0 * session_long["fraction_units"]

session_long["session_dt"] = pd.to_datetime(session_long["session_id"], errors="coerce")
if session_long["session_dt"].notna().any():
    session_long = session_long.sort_values(["session_dt", "cell_type"])
else:
    session_long = session_long.sort_values(["session_id", "cell_type"])

fig_dynamics = px.line(
    session_long,
    x="session_id",
    y="percent_units",
    color="cell_type",
    markers=True,
    title="Event-encoding prevalence across sessions (% classified units)",
    labels={
        "session_id": "Session",
        "percent_units": "% of units",
        "cell_type": "Encoding type",
    },
)
fig_dynamics.update_layout(template="plotly_white")
fig_dynamics.update_yaxes(range=[0, 100])
fig_dynamics.show()

detail_flag_map = {
    "is_choice_r1_cell": "choice_r1",
    "is_choice_r2_cell": "choice_r2",
    "is_reward_sound_r1_cell": "reward_sound_r1",
    "is_reward_sound_r2_cell": "reward_sound_r2",
}
detail_frac = (
    dfc.groupby("session_id", as_index=False)[list(detail_flag_map.keys())]
    .mean()
    .rename(columns=detail_flag_map)
)

detail_long = detail_frac.melt(
    id_vars="session_id",
    var_name="event_type",
    value_name="fraction_units",
)
detail_long["percent_units"] = 100.0 * detail_long["fraction_units"]

detail_long["session_dt"] = pd.to_datetime(detail_long["session_id"], errors="coerce")
if detail_long["session_dt"].notna().any():
    detail_long = detail_long.sort_values(["session_dt", "event_type"])
else:
    detail_long = detail_long.sort_values(["session_id", "event_type"])

fig_detail = px.line(
    detail_long,
    x="session_id",
    y="percent_units",
    color="event_type",
    markers=True,
    title="Detailed event-region prevalence across sessions (% classified units)",
    labels={
        "session_id": "Session",
        "percent_units": "% of units",
        "event_type": "Detailed event type",
    },
)
fig_detail.update_layout(template="plotly_white")
fig_detail.update_yaxes(range=[0, 100])
fig_detail.show()




Resolved interval columns: {'cue_entry': 'cue_entry_interval', 'reward1_sound': 'reward1_sound_interval', 'reward2_sound': 'reward2_sound_interval', 'R1_entry': 'R1_entry_interval', 'R2_entry': 'R2_entry_interval'}
Timing sanity check across 25 sessions (FR rows: 1,272,389, t0 rows: 19,722)


,interval_key,n_sessions,n_intervals_total,n_intervals_matched,n_intervals_unmatched,n_bins_covered,pct_intervals_matched
0,R1_entry,25,2716,2716,0,108640,100.0
1,R2_entry,25,2716,2716,0,108640,100.0
2,cue_entry,25,2716,2716,0,108640,100.0
3,reward1_sound,25,378,378,0,15120,100.0
4,reward2_sound,25,332,332,0,13280,100.0


Computed classification for 1925 unit-session entries (77 units, 25 sessions).


,n_unit_sessions
is_cue_cell,292
is_choice_cell,365
is_reward_sound_cell,248
is_choice_r1_cell,325
is_choice_r2_cell,268
is_reward_sound_r1_cell,160
is_reward_sound_r2_cell,156


,session_id,unit,n_rows_session,n_rows_cue,obs_cue,thr99_cue,p_cue,is_cue_cell,n_rows_choice,obs_choice,...,obs_reward_sound_r1,thr99_reward_sound_r1,p_reward_sound_r1,is_reward_sound_r1_cell,n_rows_reward_sound_r2,obs_reward_sound_r2,thr99_reward_sound_r2,p_reward_sound_r2,is_reward_sound_r2_cell,assigned_label
0,2024-11-14_16-40,Unit0001,32200,4080,-1.012572,0.341682,1.000000,False,8154,0.056505,...,0.893758,1.159173,0.037406,False,120,-0.161835,2.349608,0.573566,False,none
1,2024-11-14_16-40,Unit0002,32200,4080,-0.209179,0.148663,1.000000,False,8154,0.293518,...,0.144231,0.355717,0.254364,False,120,-0.275613,0.769950,0.890274,False,choice
2,2024-11-14_16-40,Unit0003,32200,4080,-0.006119,0.106145,0.598504,False,8154,0.091598,...,0.087878,0.299364,0.291771,False,120,0.191448,0.609674,0.326683,False,choice
3,2024-11-14_16-40,Unit0004,32200,4080,-0.009082,0.033017,0.865337,False,8154,0.014025,...,-0.020492,0.138123,1.000000,False,120,0.188851,0.188851,0.082294,False,none
4,2024-11-14_16-40,Unit0005,32200,4080,-0.375239,0.186081,1.000000,False,8154,0.029515,...,0.220944,0.750716,0.142145,False,120,-0.408874,1.264027,0.922693,False,none


unit-session rows: 1925 | unique units: 77 | unique sessions: 25


/var/folders/ml/fyq5x5t55wx4129dn03n32hm0000gn/T/ipykernel_53778/88185956.py:440: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



/var/folders/ml/fyq5x5t55wx4129dn03n32hm0000gn/T/ipykernel_53778/88185956.py:482: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [22]:
# Region-level and longitudinal plots (trackwise-style) + evidence heatmap
# Requires the previous event-classification cell to be executed first.

if "classification_eventwise" not in globals() or "certainty_df" not in globals():
    raise RuntimeError(
        "Run the event-classification cell first (it builds classification_eventwise and certainty_df)."
    )

MIN_SIG_SESSIONS_STABLE = 3
features = ["cue", "choice", "reward_sound"]

dfc = classification_eventwise.copy()


def _normalize_unit_from_series(series: pd.Series) -> pd.Series:
    txt = series.astype(str).str.strip()
    from_label = txt.str.extract(r"(?i)unit\s*0*(\d+)", expand=False)
    from_label_num = pd.to_numeric(from_label, errors="coerce")
    from_num = pd.to_numeric(series, errors="coerce")
    unit_num = from_label_num.combine_first(from_num)
    return unit_num.map(lambda v: f"Unit{int(v):04d}" if pd.notna(v) else np.nan)


def _build_meta_map_from_spike(spike_df: pd.DataFrame) -> pd.DataFrame:
    if spike_df is None or len(spike_df) == 0:
        return pd.DataFrame(columns=["session_id", "unit", "brain_region"])

    mdf = spike_df.reset_index().copy()
    if "session_id" not in mdf.columns:
        return pd.DataFrame(columns=["session_id", "unit", "brain_region"])

    area_candidates = ["fine_brain_area", "brain_area", "region", "area"]
    area_col = next((c for c in area_candidates if c in mdf.columns), None)
    if area_col is None:
        mdf["brain_region"] = "Unknown"
    else:
        mdf["brain_region"] = (
            mdf[area_col]
            .astype(str)
            .str.strip()
            .replace({"": np.nan, "nan": np.nan, "None": np.nan})
            .fillna("Unknown")
        )

    unit_col = None
    for candidate in ["unit", "unit_id", "unit_name", "cluster_id", "entry_id"]:
        if candidate in mdf.columns:
            u = _normalize_unit_from_series(mdf[candidate])
            if u.notna().any():
                mdf["unit"] = u
                unit_col = candidate
                break

    if "unit" not in mdf.columns:
        return pd.DataFrame(columns=["session_id", "unit", "brain_region"])

    if unit_col == "entry_id":
        shifted = _normalize_unit_from_series(pd.to_numeric(mdf["entry_id"], errors="coerce") + 1)
        n_match_default = mdf["unit"].isin(dfc["unit"].unique()).sum()
        n_match_shifted = shifted.isin(dfc["unit"].unique()).sum()
        if n_match_shifted > n_match_default:
            mdf["unit"] = shifted

    meta_map = mdf[["session_id", "unit", "brain_region"]].dropna(subset=["session_id", "unit"]).copy()
    meta_map = (
        meta_map.groupby(["session_id", "unit"], as_index=False)["brain_region"]
        .agg(lambda x: x.mode().iat[0] if not x.mode().empty else x.iloc[0])
    )
    return meta_map


meta_spike = None
if "meta_data" in globals() and isinstance(meta_data, dict) and ("SpikeClusterMetadata" in meta_data):
    meta_spike = meta_data["SpikeClusterMetadata"].copy()
else:
    try:
        if "session_names" in globals():
            meta_spike = analytics.get_analytics("SpikeClusterMetadata", session_names=session_names)
    except Exception as exc:
        print(f"Could not load SpikeClusterMetadata automatically: {exc}")

meta_map = _build_meta_map_from_spike(meta_spike)
if meta_map.empty:
    meta_map = dfc[["session_id", "unit"]].drop_duplicates().copy()
    meta_map["brain_region"] = "Unknown"

cert_small = certainty_df[
    [
        "session_id",
        "unit",
        "preferred_feature",
        "is_significant_pref",
        "pref_p",
        "certainty_score",
    ]
].copy()

row_df = (
    dfc.merge(cert_small, on=["session_id", "unit"], how="left")
    .merge(meta_map, on=["session_id", "unit"], how="left")
)
row_df["brain_region"] = row_df["brain_region"].fillna("Unknown")

row_df["encoding_state"] = np.where(
    row_df["is_significant_pref"].fillna(False),
    row_df["preferred_feature"],
    "none",
)

session_order = pd.DataFrame({"session_id": row_df["session_id"].drop_duplicates()})
session_order["session_dt"] = pd.to_datetime(
    session_order["session_id"], format="%Y-%m-%d_%H-%M", errors="coerce"
)
if session_order["session_dt"].notna().any():
    session_order = session_order.sort_values(["session_dt", "session_id"])
else:
    session_order = session_order.sort_values("session_id")
session_order["session_order"] = np.arange(len(session_order))
row_df = row_df.merge(session_order[["session_id", "session_order"]], on="session_id", how="left")


def _dominant_encoding(g: pd.DataFrame) -> str:
    sig = g[g["encoding_state"] != "none"]
    if sig.empty:
        return "none"

    counts = sig["encoding_state"].value_counts()
    top = counts[counts == counts.max()].index.tolist()
    if len(top) == 1:
        return top[0]

    tie = (
        sig[sig["encoding_state"].isin(top)]
        .groupby("encoding_state", as_index=False)["certainty_score"]
        .mean()
        .sort_values("certainty_score", ascending=False)
    )
    return tie.iloc[0]["encoding_state"]


def _unit_summary(g: pd.DataFrame) -> pd.Series:
    sig = g[g["encoding_state"] != "none"]
    n_sig = int(len(sig))
    n_unique = int(sig["encoding_state"].nunique()) if n_sig > 0 else 0

    dom = _dominant_encoding(g)
    if n_sig == 0:
        stability = "unclassified"
    elif n_unique == 1 and n_sig >= MIN_SIG_SESSIONS_STABLE:
        stability = "stable"
    elif n_unique == 1 and n_sig < MIN_SIG_SESSIONS_STABLE:
        stability = "low_support"
    else:
        stability = "changing"

    region_mode = g["brain_region"].mode()
    region = region_mode.iat[0] if not region_mode.empty else g["brain_region"].iloc[0]

    return pd.Series(
        {
            "brain_region": region,
            "dominant_encoding": dom,
            "stability_class": stability,
            "n_sessions": int(g["session_id"].nunique()),
            "n_sig_sessions": n_sig,
            "n_unique_sig_encodings": n_unique,
        }
    )


unit_profile = row_df.groupby("unit", as_index=False).apply(_unit_summary).reset_index(drop=True)

print(
    f"Units total: {len(unit_profile)} | "
    f"stable: {(unit_profile['stability_class'] == 'stable').sum()} | "
    f"changing: {(unit_profile['stability_class'] == 'changing').sum()} | "
    f"unclassified: {(unit_profile['stability_class'] == 'unclassified').sum()}"
)

# Region x dominant encoding
enc_order = ["cue", "choice", "reward_sound", "none"]
unit_profile_no_ca1 = unit_profile[unit_profile["brain_region"] != "CA1"].copy()
region_order = (
    unit_profile_no_ca1["brain_region"].value_counts().sort_values(ascending=False).index.tolist()
)

region_encoding_counts = (
    unit_profile_no_ca1.groupby(["brain_region", "dominant_encoding"], as_index=False)
    .size()
    .rename(columns={"size": "n_units"})
)

fig_region_encoding = px.bar(
    region_encoding_counts,
    x="brain_region",
    y="n_units",
    color="dominant_encoding",
    barmode="stack",
    category_orders={"dominant_encoding": enc_order, "brain_region": region_order},
    title="Neuron counts by brain region and dominant event encoding",
    labels={
        "brain_region": "Brain region",
        "n_units": "Number of neurons",
        "dominant_encoding": "Dominant encoding",
    },
)
fig_region_encoding.update_layout(template="plotly_white", xaxis_tickangle=-35)
fig_region_encoding.show()

# Region x stability
stab_order = ["stable", "changing", "low_support", "unclassified"]
region_stability_counts = (
    unit_profile_no_ca1.groupby(["brain_region", "stability_class"], as_index=False)
    .size()
    .rename(columns={"size": "n_units"})
)

fig_region_stability = px.bar(
    region_stability_counts,
    x="brain_region",
    y="n_units",
    color="stability_class",
    barmode="stack",
    category_orders={"stability_class": stab_order, "brain_region": region_order},
    title="Stable vs changing neurons by brain region (event-based)",
    labels={
        "brain_region": "Brain region",
        "n_units": "Number of neurons",
        "stability_class": "Encoding stability",
    },
)
fig_region_stability.update_layout(template="plotly_white", xaxis_tickangle=-35)
fig_region_stability.show()

# Per-neuron categorical encoding timeline
timeline = row_df[["unit", "session_id", "session_order", "encoding_state", "brain_region"]].drop_duplicates(
    subset=["unit", "session_id"]
)
ordered_sessions = session_order.sort_values("session_order")["session_id"].tolist()

stab_rank = {"changing": 0, "stable": 1, "low_support": 2, "unclassified": 3}
unit_profile_plot = unit_profile.copy()
unit_profile_plot["_stab_rank"] = unit_profile_plot["stability_class"].map(stab_rank).fillna(99)
unit_profile_plot = unit_profile_plot.sort_values(["_stab_rank", "brain_region", "unit"])
unit_order = unit_profile_plot["unit"].tolist()

state_matrix = timeline.pivot(index="unit", columns="session_id", values="encoding_state")
state_matrix = state_matrix.reindex(index=unit_order, columns=ordered_sessions)

state_to_int = {"none": 0, "cue": 1, "choice": 2, "reward_sound": 3}
z_state = state_matrix.replace(state_to_int).fillna(0).to_numpy()

label_map = unit_profile_plot.set_index("unit").apply(
    lambda r: f"{r.name} | {r['brain_region']} | {r['stability_class']}", axis=1
)
y_labels = [label_map.get(u, u) for u in state_matrix.index]

state_colorscale = [
    [0.00, "#d9d9d9"], [0.24, "#d9d9d9"],
    [0.25, "#1f77b4"], [0.49, "#1f77b4"],
    [0.50, "#ff7f0e"], [0.74, "#ff7f0e"],
    [0.75, "#2ca02c"], [1.00, "#2ca02c"],
]

fig_timeline = go.Figure(
    data=go.Heatmap(
        z=z_state,
        x=state_matrix.columns,
        y=y_labels,
        customdata=state_matrix.to_numpy(dtype=object),
        colorscale=state_colorscale,
        zmin=0,
        zmax=3,
        colorbar=dict(
            title="Encoding",
            tickvals=[0, 1, 2, 3],
            ticktext=["none", "cue", "choice", "reward_sound"],
        ),
        hovertemplate="unit=%{y}<br>session=%{x}<br>encoding=%{customdata}<extra></extra>",
    )
)

fig_timeline.update_yaxes(tickfont=dict(size=8))
fig_timeline.update_layout(
    template="plotly_white",
    title="Encoding timeline per neuron (sorted by changing/stable and brain region)",
    xaxis_title="Session",
    yaxis_title="Neuron | Region | Stability",
    height=max(500, 13 * len(y_labels)),
)
fig_timeline.update_xaxes(tickangle=-35)
fig_timeline.show()

# Evidence heatmap (continuous strength over sessions per neuron)
# strength = (obs - thr99) / |thr99| for the strongest feature in each unit-session.
ev_df = dfc[["session_id", "unit"]].copy()
strength_cols = []
for f in features:
    col = f"strength_{f}"
    ev_df[col] = (dfc[f"obs_{f}"] - dfc[f"thr99_{f}"]) / (np.abs(dfc[f"thr99_{f}"]) + 1e-12)
    strength_cols.append(col)

arr = ev_df[strength_cols].to_numpy(dtype=float)
arr_filled = np.where(np.isfinite(arr), arr, -np.inf)
has_any = np.isfinite(arr).any(axis=1)
best_idx = np.argmax(arr_filled, axis=1)
ev_df["best_feature"] = np.where(has_any, np.array(features, dtype=object)[best_idx], "none")
ev_df["best_evidence"] = np.where(has_any, arr[np.arange(len(ev_df)), best_idx], np.nan)
ev_df = ev_df.merge(
    certainty_df[["session_id", "unit", "pref_p", "is_significant_pref"]],
    on=["session_id", "unit"],
    how="left",
)

ev_matrix = (
    ev_df.pivot(index="unit", columns="session_id", values="best_evidence")
    .reindex(index=unit_order, columns=ordered_sessions)
)
feat_matrix = (
    ev_df.pivot(index="unit", columns="session_id", values="best_feature")
    .reindex(index=unit_order, columns=ordered_sessions)
    .fillna("none")
)
sig_matrix = (
    ev_df.pivot(index="unit", columns="session_id", values="is_significant_pref")
    .reindex(index=unit_order, columns=ordered_sessions)
    .fillna(False)
    .astype(bool)
)

finite_vals = ev_matrix.to_numpy(dtype=float)
finite_vals = finite_vals[np.isfinite(finite_vals)]
if finite_vals.size:
    vmax = float(np.nanpercentile(np.abs(finite_vals), 95))
    vmax = max(vmax, 0.1)
else:
    vmax = 1.0

custom_evidence = np.dstack([
    feat_matrix.to_numpy(dtype=object),
    sig_matrix.to_numpy(dtype=object),
])

fig_evidence = go.Figure(
    data=go.Heatmap(
        z=ev_matrix.to_numpy(dtype=float),
        x=ev_matrix.columns,
        y=y_labels,
        customdata=custom_evidence,
        colorscale="RdBu_r",
        zmin=-vmax,
        zmax=vmax,
        zmid=0,
        colorbar=dict(title="Best evidence (norm. obs-thr99)"),
        hovertemplate=(
            "unit=%{y}<br>session=%{x}<br>best_feature=%{customdata[0]}"
            "<br>is_significant=%{customdata[1]}<br>evidence=%{z:.3f}<extra></extra>"
        ),
    )
)
fig_evidence.update_yaxes(tickfont=dict(size=8))
fig_evidence.update_layout(
    template="plotly_white",
    title="Encoding evidence heatmap per neuron over sessions",
    xaxis_title="Session",
    yaxis_title="Neuron | Region | Stability",
    height=max(500, 13 * len(y_labels)),
)
fig_evidence.update_xaxes(tickangle=-35)
fig_evidence.show()

# Multi-encoding longitudinal analysis (broad event classes)
multi_flag_cols = {
    "cue": "is_cue_cell",
    "choice": "is_choice_cell",
    "reward_sound": "is_reward_sound_cell",
}
missing_cols = [c for c in multi_flag_cols.values() if c not in row_df.columns]
if missing_cols:
    raise KeyError(f"Missing multi-encoding columns in row_df: {missing_cols}")

multi_timeline = row_df[
    ["unit", "session_id", "session_order", "brain_region", *multi_flag_cols.values()]
].drop_duplicates(subset=["unit", "session_id"]).copy()

for col in multi_flag_cols.values():
    multi_timeline[col] = multi_timeline[col].fillna(False).astype(bool)


def _combo_from_flags(r: pd.Series) -> str:
    labels = [name for name, col in multi_flag_cols.items() if bool(r[col])]
    return "+".join(labels) if labels else "none"


multi_timeline["multi_encoding_label"] = multi_timeline.apply(_combo_from_flags, axis=1)
multi_timeline["n_zone_encodings"] = (
    multi_timeline[list(multi_flag_cols.values())].sum(axis=1).astype(int)
)
multi_timeline["is_multi_encoding"] = multi_timeline["n_zone_encodings"] >= 2


def _count_transitions(values) -> int:
    vals = list(values)
    return int(sum(a != b for a, b in zip(vals[:-1], vals[1:])))


def _multi_unit_summary(g: pd.DataFrame) -> pd.Series:
    if "session_order" in g.columns and g["session_order"].notna().any():
        g_sorted = g.sort_values(["session_order", "session_id"])
    else:
        g_sorted = g.sort_values("session_id")

    n_total = int(g_sorted["session_id"].nunique())
    n_multi = int(g_sorted["is_multi_encoding"].sum())
    n_any = int((g_sorted["n_zone_encodings"] >= 1).sum())
    multi_only = g_sorted[g_sorted["is_multi_encoding"]]

    return pd.Series(
        {
            "n_sessions_total": n_total,
            "n_any_encoding_sessions": n_any,
            "n_multi_sessions": n_multi,
            "frac_multi_sessions": (n_multi / n_total) if n_total else np.nan,
            "ever_multi_encoded": bool(n_multi > 0),
            "n_unique_multi_labels": int(multi_only["multi_encoding_label"].nunique()) if n_multi else 0,
            "label_transitions_all_sessions": _count_transitions(g_sorted["multi_encoding_label"].tolist()),
            "label_transitions_multi_sessions": _count_transitions(multi_only["multi_encoding_label"].tolist()),
        }
    )


multi_unit_profile = multi_timeline.groupby("unit").apply(_multi_unit_summary).reset_index()
multi_unit_profile = unit_profile.merge(multi_unit_profile, on="unit", how="left")
multi_unit_profile["ever_multi_encoded"] = multi_unit_profile["ever_multi_encoded"].fillna(False).astype(bool)
for col in [
    "n_sessions_total",
    "n_any_encoding_sessions",
    "n_multi_sessions",
    "n_unique_multi_labels",
    "label_transitions_all_sessions",
    "label_transitions_multi_sessions",
]:
    if col in multi_unit_profile.columns:
        multi_unit_profile[col] = multi_unit_profile[col].fillna(0).astype(int)
multi_unit_profile["frac_multi_sessions"] = multi_unit_profile["frac_multi_sessions"].fillna(0.0)

n_total_units = int(multi_unit_profile["unit"].nunique())
n_ever_multi = int(multi_unit_profile["ever_multi_encoded"].sum())
print(
    f"Units with multi-zone encoding in at least one session: {n_ever_multi}/{n_total_units} "
    f"({100 * n_ever_multi / max(n_total_units, 1):.1f}%)"
)

session_multi = (
    multi_timeline.groupby(["session_id", "session_order"], as_index=False)
    .agg(
        n_units=("unit", "nunique"),
        n_multi_units=("is_multi_encoding", "sum"),
        mean_n_zone_encodings=("n_zone_encodings", "mean"),
    )
    .sort_values(["session_order", "session_id"])
)
session_multi["pct_multi_units"] = 100.0 * session_multi["n_multi_units"] / session_multi["n_units"].clip(lower=1)

fig_multi_dynamics = px.line(
    session_multi,
    x="session_id",
    y="pct_multi_units",
    markers=True,
    title="Multi-encoding prevalence across sessions (% neurons with >=2 encodings)",
    labels={
        "session_id": "Session",
        "pct_multi_units": "% of neurons with >=2 encodings",
    },
)
fig_multi_dynamics.update_layout(template="plotly_white")
fig_multi_dynamics.update_yaxes(range=[0, max(5, min(100, session_multi["pct_multi_units"].max() * 1.15))])
fig_multi_dynamics.update_xaxes(tickangle=-35)
fig_multi_dynamics.show()

if n_ever_multi == 0:
    print("Skipping multi-encoding timeline heatmap because no neurons showed >=2 simultaneous encodings.")
else:
    multi_plot_profile = multi_unit_profile[multi_unit_profile["ever_multi_encoded"]].copy()
    multi_plot_profile["_stab_rank"] = multi_plot_profile["stability_class"].map(stab_rank).fillna(99)
    multi_plot_profile = multi_plot_profile.sort_values(
        ["_stab_rank", "brain_region", "n_multi_sessions", "unit"],
        ascending=[True, True, False, True],
    )
    multi_unit_order = multi_plot_profile["unit"].tolist()

    combo_matrix = (
        multi_timeline.pivot(index="unit", columns="session_id", values="multi_encoding_label")
        .reindex(index=multi_unit_order, columns=ordered_sessions)
        .fillna("none")
    )
    nzone_matrix = (
        multi_timeline.pivot(index="unit", columns="session_id", values="n_zone_encodings")
        .reindex(index=multi_unit_order, columns=ordered_sessions)
        .fillna(0)
        .astype(int)
    )

    combo_order = [
        "none",
        "cue",
        "choice",
        "reward_sound",
        "cue+choice",
        "cue+reward_sound",
        "choice+reward_sound",
        "cue+choice+reward_sound",
    ]
    combo_colors = {
        "none": "#d9d9d9",
        "cue": "#1f77b4",
        "choice": "#ff7f0e",
        "reward_sound": "#2ca02c",
        "cue+choice": "#9467bd",
        "cue+reward_sound": "#17becf",
        "choice+reward_sound": "#bcbd22",
        "cue+choice+reward_sound": "#d62728",
    }

    def _discrete_colorscale(colors):
        n = len(colors)
        if n == 1:
            return [[0.0, colors[0]], [1.0, colors[0]]]
        cs = []
        for i, color in enumerate(colors):
            cs.append([i / n, color])
            cs.append([(i + 1) / n, color])
        return cs

    combo_to_z = {label: i + 0.5 for i, label in enumerate(combo_order)}
    z_combo = combo_matrix.replace(combo_to_z).to_numpy(dtype=float)

    multi_label_map = multi_plot_profile.set_index("unit").apply(
        lambda r: (
            f"{r.name} | {r['brain_region']} | {r['stability_class']} | "
            f"multi={int(r['n_multi_sessions'])}"
        ),
        axis=1,
    )
    y_multi = [multi_label_map.get(u, u) for u in combo_matrix.index]

    customdata = np.dstack([
        combo_matrix.to_numpy(dtype=object),
        nzone_matrix.to_numpy(dtype=object),
    ])

    fig_multi_timeline = go.Figure(
        data=go.Heatmap(
            z=z_combo,
            x=combo_matrix.columns,
            y=y_multi,
            customdata=customdata,
            colorscale=_discrete_colorscale([combo_colors[k] for k in combo_order]),
            zmin=0,
            zmax=len(combo_order),
            colorbar=dict(
                title="Encoding combo",
                tickvals=[i + 0.5 for i in range(len(combo_order))],
                ticktext=combo_order,
            ),
            hovertemplate=(
                "unit=%{y}<br>session=%{x}<br>encoding_combo=%{customdata[0]}"
                "<br>n_zone_encodings=%{customdata[1]}<extra></extra>"
            ),
        )
    )

    fig_multi_timeline.update_yaxes(tickfont=dict(size=8))
    fig_multi_timeline.update_layout(
        template="plotly_white",
        title=(
            "Multi-encoding timeline per neuron (ever multi-encoded; sorted by changing/stable and brain region)"
        ),
        xaxis_title="Session",
        yaxis_title="Neuron | Region | Stability | # multi sessions",
        height=max(550, 14 * len(y_multi)),
    )
    fig_multi_timeline.update_xaxes(tickangle=-35)
    fig_multi_timeline.show()

# Optional quick lookup tables
stable_units = unit_profile[unit_profile["stability_class"] == "stable"].sort_values(["brain_region", "unit"])
changing_units = unit_profile[unit_profile["stability_class"] == "changing"].sort_values(["brain_region", "unit"])

display(stable_units.head(20))
display(changing_units.head(20))




2026-03-02 16:33:53,814|DEBUG|53778|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
Inferring NAS paths for list of session names...================================================================================

Inferring NAS paths for list of session names...================================================================================

2026-03-02 16:33:54,335|DEBUG|53778|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-03-02 16:33:54,339|DEBUG|53778|analytics|get_analytics
	Processing SpikeClusterMetadata, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hd

Units total: 77 | stable: 16 | changing: 53 | unclassified: 3


/var/folders/ml/fyq5x5t55wx4129dn03n32hm0000gn/T/ipykernel_53778/2636628195.py:171: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



/var/folders/ml/fyq5x5t55wx4129dn03n32hm0000gn/T/ipykernel_53778/2636628195.py:251: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



Units with multi-zone encoding in at least one session: 42/77 (54.5%)


/var/folders/ml/fyq5x5t55wx4129dn03n32hm0000gn/T/ipykernel_53778/2636628195.py:432: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



/var/folders/ml/fyq5x5t55wx4129dn03n32hm0000gn/T/ipykernel_53778/2636628195.py:536: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



,unit,brain_region,dominant_encoding,stability_class,n_sessions,n_sig_sessions,n_unique_sig_encodings
21,Unit0022,ACC,choice,stable,25,13,1
1,Unit0002,CA1,choice,stable,25,13,1
3,Unit0004,DG,choice,stable,25,3,1
7,Unit0008,DG,cue,stable,25,10,1
12,Unit0013,DG,choice,stable,25,3,1
14,Unit0015,DG,cue,stable,25,24,1
16,Unit0017,DG,reward_sound,stable,25,10,1
17,Unit0018,DG,reward_sound,stable,25,8,1
76,Unit0077,InfrL,reward_sound,stable,25,3,1
36,Unit0037,PrL,reward_sound,stable,25,5,1


,unit,brain_region,dominant_encoding,stability_class,n_sessions,n_sig_sessions,n_unique_sig_encodings
20,Unit0021,ACC,choice,changing,25,10,2
22,Unit0023,ACC,choice,changing,25,5,3
23,Unit0024,ACC,choice,changing,25,24,3
24,Unit0025,ACC,choice,changing,25,16,3
25,Unit0026,ACC,choice,changing,25,10,2
26,Unit0027,ACC,reward_sound,changing,25,17,2
27,Unit0028,ACC,choice,changing,25,2,2
0,Unit0001,CA1,choice,changing,25,10,2
4,Unit0005,DG,choice,changing,25,2,2
5,Unit0006,DG,reward_sound,changing,25,12,2
